In [19]:
# ================================================================
# 감정 분류 LoRA 모델 훈련 & 추론
# Colab T4 GPU 기준 / klue/bert-base + peft
#
# [사용 방법]
# 1. Colab에서 런타임 → GPU (T4) 로 변경
# 2. 셀을 위에서 아래로 순서대로 실행
# 3. CSV 파일을 Colab에 업로드하거나 Drive 경로 수정
# ================================================================

In [20]:
!pip uninstall -y bitsandbytes torchao

In [2]:
%pip uninstall -y bitsandbytes torchao
%pip install -U transformers peft accelerate datasets evaluate scikit-learn

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: dataset

In [3]:
import importlib.util
# 두 패키지 다 없어야 가능
print("bitsandbytes:", importlib.util.find_spec("bitsandbytes"))
print("torchao:", importlib.util.find_spec("torchao"))

bitsandbytes: None
torchao: None


In [22]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
# ============================================================
# LoRA 기반 한국어 감정 분류 모델 학습 코드
# 데이터: emotion_final.csv
# 입력: 사람문장
# 출력: emotion 감정 라벨
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)


In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
# ============================================================
# 1. 기본 설정
# ============================================================

# 데이터 파일 경로
# Colab이나 로컬에서 실행할 경우 본인 경로에 맞게 수정하면 됩니다.
DATA_PATH = "emotion_final.csv"

# 사용할 한국어 사전학습 모델
# klue/roberta-base는 한국어 분류 작업에 무난하게 사용할 수 있습니다.
BASE_MODEL_NAME = "klue/roberta-base"

# 학습 결과 저장 폴더
OUTPUT_DIR = "./emotion_lora_model"

# 문장 컬럼명
TEXT_COL = "사람문장"

# 감정 라벨 컬럼명
LABEL_COL = "emotion"

# 최대 토큰 길이
# 감정 문장이 짧은 편이면 128 정도면 충분합니다.
MAX_LENGTH = 128

# 학습 관련 설정
NUM_EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-4
SEED = 42

In [7]:
# ============================================================
# 2. 재현성을 위한 시드 고정
# ============================================================

def set_seed(seed: int = 42):
    """
    매번 실행할 때 결과가 최대한 비슷하게 나오도록
    random, numpy, torch의 seed를 고정합니다.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [8]:
# 모델 학습 파라미터
MAX_LEN      = 128    # 토큰 최대 길이 (사람문장 3개 합치면 128이면 충분)
BATCH_SIZE   = 8      # T4 기준 안전한 크기 (OOM 나면 4로 줄이기)
GRAD_ACCUM   = 4      # 실질 배치 = BATCH_SIZE × GRAD_ACCUM = 32
EPOCHS       = 5
LR           = 2e-4   # LoRA는 일반 파인튜닝보다 높은 lr 사용
TEST_SIZE    = 0.1    # 전체의 10% = 약 5,100건을 검증셋으로

# ── LoRA 설정 ──────────────────────────────
LORA_R       = 8      # rank: 낮을수록 가볍고 빠름 (8~16 권장)
LORA_ALPHA   = 16
LORA_DROPOUT = 0.1

In [9]:
# ============================================================
# 3. 데이터 불러오기
# ============================================================

df = pd.read_csv(DATA_PATH)

print("데이터 크기:", df.shape)
print("컬럼 목록:", df.columns.tolist())
print(df.head())

데이터 크기: (51630, 5)
컬럼 목록: ['연령', '성별', '상황키워드', '사람문장', 'emotion']
   연령  성별     상황키워드                                               사람문장  \
0  청년  여성  진로,취업,직장  일은 왜 해도 해도 끝이 없을까? 화가 난다. 그냥 내가 해결하는 게 나아. 남들한...   
1  청년  여성  진로,취업,직장  이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나....   
2  청년  여성  진로,취업,직장  회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...   
3  청년  여성  진로,취업,직장  직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...   
4  청년  여성  진로,취업,직장  얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나. 상사인 나에게 ...   

    emotion  
0  분노_노여워하는  
1  분노_노여워하는  
2  분노_노여워하는  
3  분노_노여워하는  
4  분노_노여워하는  


In [10]:
# ============================================================
# 4. 필요한 컬럼만 사용하고 결측치 제거
# ============================================================

# 사람문장과 emotion 컬럼만 사용합니다.
df = df[[TEXT_COL, LABEL_COL]].copy()

# 결측치 제거
df = df.dropna(subset=[TEXT_COL, LABEL_COL])

# 텍스트를 문자열로 변환
df[TEXT_COL] = df[TEXT_COL].astype(str)

# 라벨도 문자열로 변환
df[LABEL_COL] = df[LABEL_COL].astype(str)

print("전처리 후 데이터 크기:", df.shape)
print("감정 라벨 개수:", df[LABEL_COL].nunique())
print(df[LABEL_COL].value_counts().head())

전처리 후 데이터 크기: (51630, 2)
감정 라벨 개수: 60
emotion
불안_걱정스러운     1229
분노_짜증내는      1054
슬픔_우울한       1002
슬픔_눈물이 나는     993
불안_두려운        991
Name: count, dtype: int64


In [11]:
# ============================================================
# 5. 라벨 인코딩
# ============================================================

"""
딥러닝 모델은 '분노_노여워하는' 같은 문자열 라벨을 직접 학습할 수 없습니다.
그래서 각 감정 라벨을 숫자로 바꿉니다.

예:
분노_노여워하는 -> 0
불안_걱정스러운 -> 1
슬픔_우울한 -> 2
...
"""

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df[LABEL_COL])

num_labels = len(label_encoder.classes_)

print("라벨 수:", num_labels)
print("라벨 예시:")
for i, label_name in enumerate(label_encoder.classes_[:10]):
    print(i, ":", label_name)


# 라벨 번호와 이름을 저장해 둡니다.
# 나중에 앱에서 예측 결과 숫자를 감정 이름으로 바꿀 때 필요합니다.
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "id2label": id2label,
            "label2id": label2id
        },
        f,
        ensure_ascii=False,
        indent=2
    )


라벨 수: 60
라벨 예시:
0 : 기쁨_감사하는
1 : 기쁨_기쁨
2 : 기쁨_느긋
3 : 기쁨_만족스러운
4 : 기쁨_신뢰하는
5 : 기쁨_신이 난
6 : 기쁨_안도
7 : 기쁨_자신하는
8 : 기쁨_편안한
9 : 기쁨_흥분


In [12]:
# ============================================================
# 6. 학습 데이터 / 검증 데이터 분리
# ============================================================

"""
stratify=df["label"]을 넣으면
학습 데이터와 검증 데이터에 감정 라벨 비율이 비슷하게 나뉩니다.

감정 데이터처럼 클래스가 많은 분류 문제에서는
라벨 분포를 유지하는 것이 중요합니다.
"""

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

print("학습 데이터:", train_df.shape)
print("검증 데이터:", valid_df.shape)

학습 데이터: (41304, 3)
검증 데이터: (10326, 3)


In [13]:
# ============================================================
# 7. Hugging Face Dataset 형태로 변환
# ============================================================

"""
transformers의 Trainer를 사용하기 위해
pandas DataFrame을 Dataset 형태로 바꿉니다.
"""

train_dataset = Dataset.from_pandas(train_df[[TEXT_COL, "label"]])
valid_dataset = Dataset.from_pandas(valid_df[[TEXT_COL, "label"]])

In [14]:
# ============================================================
# 8. 토크나이저 불러오기
# ============================================================

"""
토크나이저는 문장을 모델이 이해할 수 있는 숫자 토큰으로 바꿉니다.
한국어 문장:
"오늘 너무 화가 난다"
-> [2, 3842, 1231, ...] 같은 숫자 배열
"""

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)


def tokenize_function(batch):
    """
    문장을 토큰화하는 함수입니다.
    truncation=True:
        문장이 너무 길면 MAX_LENGTH에 맞게 자릅니다.

    max_length=MAX_LENGTH:
        최대 토큰 길이를 지정합니다.

    padding은 여기서 하지 않습니다.
    DataCollatorWithPadding이 배치마다 동적으로 패딩합니다.
    """
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)


# Trainer가 사용할 수 있도록 컬럼 정리
train_dataset = train_dataset.remove_columns([TEXT_COL, "__index_level_0__"])
valid_dataset = valid_dataset.remove_columns([TEXT_COL, "__index_level_0__"])

# PyTorch tensor 형태로 설정
train_dataset.set_format("torch")
valid_dataset.set_format("torch")

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Map:   0%|          | 0/41304 [00:00<?, ? examples/s]

Map:   0%|          | 0/10326 [00:00<?, ? examples/s]

In [15]:
# ============================================================
# 9. 기본 분류 모델 불러오기
# ============================================================

"""
AutoModelForSequenceClassification은 문장 분류용 모델입니다.

num_labels:
    감정 라벨 개수입니다. 현재 데이터에서는 60개입니다.

id2label, label2id:
    모델 내부에 라벨 이름 정보를 저장합니다.
    나중에 예측 결과를 해석할 때 편리합니다.
"""

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
# ============================================================
# 10. LoRA 설정
# ============================================================

"""
LoRA는 전체 모델을 다 학습하지 않고,
일부 작은 어댑터 파라미터만 학습하는 방식입니다.

장점:
1. 학습 속도가 빠름
2. GPU 메모리를 적게 사용함
3. 저장되는 모델 크기가 작음

BERT/RoBERTa 계열 모델에서는 보통 attention의 query, value에 LoRA를 적용합니다.
"""

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,      # 문장 분류 작업
    r=8,                             # LoRA rank. 클수록 표현력 증가, 메모리 증가
    lora_alpha=16,                   # LoRA scaling 값
    lora_dropout=0.1,                # 과적합 방지용 dropout
    target_modules=["query", "value"] # RoBERTa attention 내부 모듈
)

# 기존 모델에 LoRA 어댑터를 붙입니다.
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 수 확인
model.print_trainable_parameters()


trainable params: 931,644 || all params: 111,595,896 || trainable%: 0.8348


In [17]:
# ============================================================
# 11. 평가 지표 정의
# ============================================================

def compute_metrics(eval_pred):
    """
    검증 데이터 평가 시 사용할 지표입니다.

    accuracy:
        전체 중 맞춘 비율

    macro_f1:
        각 감정 라벨의 F1 점수를 동일한 비중으로 평균낸 값
        클래스가 많은 감정 분류에서는 accuracy보다 macro_f1도 같이 보는 것이 좋습니다.
    """
    logits, labels = eval_pred

    # logits에서 가장 높은 값을 가진 라벨을 예측값으로 사용
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [18]:
# ============================================================
# 12. Data Collator 설정
# ============================================================

"""
DataCollatorWithPadding은 배치마다 가장 긴 문장에 맞춰 padding을 해줍니다.

예:
배치 안 문장 길이가 [20, 35, 12]라면
모두 길이 35로 맞춥니다.

이 방식은 모든 문장을 무조건 MAX_LENGTH로 padding하는 것보다 효율적입니다.
"""

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# ============================================================
# 13. TrainingArguments 설정
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # 학습 횟수
    num_train_epochs=NUM_EPOCHS,

    # 배치 크기
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    # 학습률
    learning_rate=LEARNING_RATE,

    # weight decay는 과적합 방지에 도움
    weight_decay=0.01,

    # 평가와 저장 전략
    eval_strategy="epoch",
    save_strategy="epoch",

    # 가장 좋은 모델을 마지막에 불러오기
    load_best_model_at_end=True,

    # 좋은 모델 판단 기준
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # 로그 출력 주기
    logging_steps=100,

    # 저장할 체크포인트 개수 제한
    save_total_limit=2,

    # GPU 사용 시 fp16 사용
    # GPU가 없거나 오류가 나면 False로 바꾸세요.
    fp16=torch.cuda.is_available(),

    # 결과 재현성
    seed=SEED,

    # 외부 로깅 사용 안 함
    report_to="none"
)


In [20]:
# ============================================================
# 14. Trainer 생성
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [21]:
# ============================================================
# 15. 모델 학습
# ============================================================
trainer.train()
# ============================================================
# 16. 최종 평가
# ============================================================
eval_result = trainer.evaluate()

print("최종 평가 결과")
print(eval_result)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.401312,2.371071,0.401995,0.396972,0.392742
2,2.236724,2.288380,0.427949,0.425556,0.421504
3,2.178616,2.236249,0.432694,0.433232,0.428918


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
2.178616,2.236249,3,0.432694,0.433232,0.428918


최종 평가 결과
{'eval_loss': 2.2362494468688965, 'eval_accuracy': 0.4326941700561689, 'eval_macro_f1': 0.43323179656347593, 'eval_weighted_f1': 0.42891773504544006}


In [22]:
# ============================================================
# 17. 모델 저장
# ============================================================

"""
LoRA 모델은 base model 전체가 아니라
LoRA adapter 중심으로 저장됩니다.

저장되는 주요 파일:
- adapter_model.safetensors
- adapter_config.json
- tokenizer 파일들
- label_map.json
"""

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"모델 저장 완료: {OUTPUT_DIR}")

모델 저장 완료: ./emotion_lora_model
